In [ ]:
# Install requirements to call Qualcomm AI Inference Suite via Python SDK
!pip install imagine_sdk-0.4.2-py3-none-any.whl

# Install faiss to use for searching similarity of our query to available data
!pip install faiss-cpu


In [ ]:
# import required libraries
from imagine import ChatMessage, ImagineClient, ModelType
from pprint import pprint
import faiss, numpy as np

In [ ]:
# Retrieve stored endpoint and API key for use
from google.colab import userdata

# store endpoint and API key
myendpoint = userdata.get('ENDPOINT')
myapikey = userdata.get('API_KEY')

In [ ]:
# create some data to test embedding
documents = [
    "Ray works at Qualcomm.",
    "Ray is a specialist in developer relations.",
    "Qualcomm AI Inference Suite is great for AI inference workloads",
]
pprint(documents)

In [ ]:
# set up our client
client = ImagineClient(myendpoint, myapikey)

In [ ]:
# check installed models and grab one to use
all_models = client.get_available_models_by_type()
embedding_models = client.get_available_models_by_type(ModelType.EMBEDDING)
use_embed_model = embedding_models.get(ModelType.EMBEDDING, 0)[1]
pprint(use_embed_model)

In [ ]:
# create embeddings
doc_embeddings = client.embeddings(documents, model=use_embed_model)
doc_embeddings = doc_embeddings.data
for item in doc_embeddings:
    pprint(item)

In [ ]:
# get embeddings into the right format for FAISS use
embeddings_only = []
for item in doc_embeddings:
    embeddings_only.append(item.embedding)
doc_embeddings = np.array(embeddings_only).astype("float32")
pprint(doc_embeddings.shape)


In [ ]:
# create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
# query and retrieve relevant document - example 1
query = "How can I do AI inference?"
query_embedding = client.embeddings([query], model=use_embed_model).data[0].embedding

query_vector = np.array(query_embedding).astype("float32")
query_vector = query_vector.reshape(1, -1) # Reshape to (1, dimension)
D, I = index.search(query_vector, k=1)

retrieved_doc = documents[I[0][0]]
pprint(retrieved_doc)

In [ ]:
# query and retrieve relevant document - example 2
query = "Who is Ray?"
query_embedding = client.embeddings([query], model=use_embed_model).data[0].embedding

query_vector = np.array(query_embedding).astype("float32")
query_vector = query_vector.reshape(1, -1) # Reshape to (1, dimension)
D, I = index.search(query_vector, k=1)

retrieved_doc = documents[I[0][0]]
pprint(retrieved_doc)

In [ ]:
# Let's call an LLM to have it answer with the provided data
payload = {
    "model": "Llama-3.1-8B", # can try other models as well
    "messages": [
        {"role": "system", "content": "Answer the question using the provided context.  If you can't answer using the provided context, say that the data is not in the document set."},
        {"role": "user", "content": f"Context: {retrieved_doc}\nQuestion: {query}"}
    ]
}
chat_response = client.chat(messages=payload["messages"], model=payload["model"])
pprint(chat_response.first_content)